# 🎬 Netflix User Segmentation — K-Means Clustering & PCA

**Session 3 | Industrial Use Case 1 | DSA & ML for Business**

---

### Business Context
- Netflix has **247M subscribers** across **190 countries**
- Content library costs **$17B/year** — ROI depends on matching content to segments
- Netflix loses **$1.6B/year** to churn (avg subscriber stays 15 months)
- **80% of content watched** comes from algorithmic recommendations

### What You'll Learn
1. **Feature engineering** — encode categorical features for clustering
2. **K-Means clustering** with Elbow & Silhouette validation
3. **PCA visualization** — reduce dimensions for 2D cluster plots
4. **Cluster profiling** — name each segment with business personas
5. **Retention strategy** — connect segments to actionable business decisions

## Step 1: Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

df = pd.read_csv("../datasets/netflix_user_segmentation.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nSubscription types: {df['subscription_type'].value_counts().to_dict()}")
print(f"Device types: {df['device_primary'].value_counts().to_dict()}")
df.describe().round(2)

## Step 2: Feature Engineering & Scaling

Encode categorical features and scale all features — **critical for K-Means** since it uses Euclidean distance.

In [ ]:
# Encode categorical features
sub_order = {'ad_supported': 0, 'basic': 1, 'standard': 2, 'premium': 3}
df['subscription_encoded'] = df['subscription_type'].map(sub_order)

device_dummies = pd.get_dummies(df['device_primary'], prefix='device')
df_features = pd.concat([
    df[['avg_watch_hours', 'genre_diversity_score', 'days_active', 'completion_rate', 'subscription_encoded']],
    device_dummies
], axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features)
feature_names = df_features.columns.tolist()

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Features: {feature_names}")

## Step 3: Determine Optimal K — Elbow Method & Silhouette Score

In [ ]:
K_range = range(2, 11)
inertias = []
silhouette_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))
    print(f"K={k:2d} | Inertia: {kmeans.inertia_:>10,.1f} | Silhouette: {silhouette_score(X_scaled, labels):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-', linewidth=2)
axes[0].set_xlabel('Number of Clusters (K)'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method', fontsize=14, fontweight='bold'); axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2)
axes[1].set_xlabel('Number of Clusters (K)'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score by K', fontsize=14, fontweight='bold'); axes[1].grid(True, alpha=0.3)

optimal_k = list(K_range)[np.argmax(silhouette_scores)]
axes[1].axvline(x=optimal_k, color='red', linestyle='--', alpha=0.7, label=f'Best K={optimal_k}')
axes[1].legend(fontsize=11)
plt.tight_layout()
plt.show()
print(f"\n✅ Optimal K by Silhouette Score: {optimal_k}")

## Step 4: Run K-Means with K=5 & Profile Clusters

In [ ]:
K = 5
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

print(f"K-Means with K={K} | Silhouette: {silhouette_score(X_scaled, df['cluster']):.4f}")
print(f"\nCluster sizes:")
print(df['cluster'].value_counts().sort_index())

profile = df.groupby('cluster').agg({
    'avg_watch_hours': 'mean',
    'genre_diversity_score': 'mean',
    'subscription_type': lambda x: x.mode()[0],
    'days_active': 'mean',
    'completion_rate': 'mean',
    'device_primary': lambda x: x.mode()[0],
}).round(3)

persona_names = {}
for cluster_id in range(K):
    c = profile.loc[cluster_id]
    if c['avg_watch_hours'] > 4 and c['days_active'] > 20:
        persona_names[cluster_id] = '🔥 Binge Watcher'
    elif c['avg_watch_hours'] < 2 and c['days_active'] < 12:
        persona_names[cluster_id] = '👀 Casual Browser'
    elif c['device_primary'] == 'mobile' and c['days_active'] > 15:
        persona_names[cluster_id] = '📱 Mobile Streamer'
    elif c['genre_diversity_score'] > 0.5 and c['subscription_type'] == 'premium':
        persona_names[cluster_id] = '💎 Premium Enthusiast'
    else:
        persona_names[cluster_id] = '📅 Weekend Warrior'

profile['persona'] = [persona_names[i] for i in range(K)]
print("\n=== Cluster Profiles ===")
print(profile.to_string())

## Step 5: PCA Visualization

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA explained variance: {pca.explained_variance_ratio_}")
print(f"Total variance retained: {sum(pca.explained_variance_ratio_):.1%}")

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.Set2.colors
for cluster_id in range(K):
    mask = df['cluster'] == cluster_id
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=[colors[cluster_id]], label=persona_names[cluster_id], alpha=0.5, s=20)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Netflix User Segments (PCA Projection)', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, markerscale=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 6: Strategic Analysis — Segment Strategy & Revenue Impact

In [ ]:
pricing = {'ad_supported': 6.99, 'basic': 9.99, 'standard': 15.49, 'premium': 22.99}
segment_ltv = df.groupby('cluster').apply(
    lambda x: pd.Series({
        'size': len(x),
        'pct': len(x) / len(df) * 100,
        'avg_watch_hrs': x['avg_watch_hours'].mean(),
        'avg_days_active': x['days_active'].mean(),
        'top_sub': x['subscription_type'].mode()[0],
        'avg_monthly_value': x['subscription_type'].map(pricing).mean(),
        'completion': x['completion_rate'].mean(),
        'churn_risk': 1 - x['days_active'].mean() / 30
    })
)
segment_ltv['persona'] = [persona_names[i] for i in range(K)]
segment_ltv['annual_revenue_per_user'] = segment_ltv['avg_monthly_value'] * 12
segment_ltv['est_ltv'] = segment_ltv['annual_revenue_per_user'] * (1 / (segment_ltv['churn_risk'] + 0.1))

print("=" * 70)
print("SEGMENT STRATEGY BRIEF")
print("=" * 70)
for i in range(K):
    s = segment_ltv.loc[i]
    print(f"\n{'─'*50}")
    print(f"SEGMENT {i+1}: {s['persona']}")
    print(f"  Size: {s['size']:.0f} users ({s['pct']:.1f}%)")
    print(f"  Avg Watch: {s['avg_watch_hrs']:.1f} hrs/day | Active: {s['avg_days_active']:.0f} days/month")
    print(f"  Completion Rate: {s['completion']:.1%} | Churn Risk: {s['churn_risk']:.1%}")
    print(f"  Monthly Revenue: ${s['avg_monthly_value']:.2f} | Est. LTV: ${s['est_ltv']:.0f}")

best = segment_ltv.nlargest(1, 'est_ltv').iloc[0]
worst = segment_ltv.nlargest(1, 'churn_risk').iloc[0]
print(f"\n💎 HIGHEST LTV: {best['persona']} (${best['est_ltv']:.0f})")
print(f"⚠️ HIGHEST CHURN RISK: {worst['persona']} ({worst['churn_risk']:.1%})")

at_risk_users = worst['size'] * worst['churn_risk']
saved_users = at_risk_users * 0.05
revenue_saved = saved_users * worst['annual_revenue_per_user']
print(f"\n💰 RETENTION IMPROVEMENT (5% in at-risk segment):")
print(f"   Users saved: {saved_users:.0f}")
print(f"   Annual revenue saved: ${revenue_saved:,.0f}")
print(f"   At Netflix scale (247M users): ${revenue_saved * (247_000_000 / len(df)):,.0f}")